# CV-6 — validação técnica no Google Colab

Tech Challenge — Fase 1, FIAP. Validar Dataset, DataLoaders e MobileNetV3 Small,
sem treinamento, épocas, métricas preditivas ou pesos baixados. Comece com dados
**sintéticos temporários**; dados reais e Drive são opcionais.

Este notebook é entregue **sem execução dinâmica registrada**. Saídas vazias não
comprovam aprovação. Execute em ordem e preserve o resumo ao final. Nenhum logit
de uma rede sem treinamento representa uma previsão clínica válida.

No Colab: Arquivo → Fazer upload de notebook. CPU basta para a primeira execução;
GPU é opcional. As instruções completas estão em `docs/vision_colab_validation.md`.

## 1. Ambiente e versões existentes

A primeira inspeção não instala nada. A compatibilidade é testada em subprocesso,
incluindo importação e uma operação CPU de torchvision. Isso evita manter imports
antigos no kernel caso seja necessário corrigir o runtime.

In [ ]:
import importlib.metadata as metadata
import json
import platform
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
print("Sistema:", platform.platform())
for package in ("torch", "torchvision", "numpy", "pandas", "Pillow", "scikit-learn"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "ausente")

PROBE_SOURCE = r"""
import importlib, json
info = {"torch_compatible": False, "dependencies": {}, "errors": {}}
try:
    import torch, torchvision
    torchvision.ops.nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5)
    info.update(torch_compatible=True, torch=torch.__version__, torchvision=torchvision.__version__,
                cuda_build=torch.version.cuda, cuda_available=torch.cuda.is_available(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except Exception as exc:
    info["errors"]["torch_stack"] = repr(exc)
for module in ("numpy", "pandas", "PIL", "sklearn"):
    try:
        loaded = importlib.import_module(module)
        info["dependencies"][module] = getattr(loaded, "__version__", "importado")
    except Exception as exc:
        info["errors"][module] = repr(exc)
info["ok"] = info["torch_compatible"] and len(info["dependencies"]) == 4
print(json.dumps(info))
"""

def probe_environment():
    probe = subprocess.run([sys.executable, "-c", PROBE_SOURCE], capture_output=True,
                           text=True, timeout=120)
    if probe.returncode:
        raise RuntimeError("Falha na inspeção: " + probe.stderr)
    if probe.stderr.strip():
        print("Avisos da inspeção:", probe.stderr.strip())
    return json.loads(probe.stdout.strip().splitlines()[-1])

environment = probe_environment()
print(json.dumps(environment, indent=2))

## 2. Disponibilizar o código atual

Escolha **uma** opção em `CODE_SOURCE`:

- `upload_zip` (padrão): enviar `tech-challenge-cv6-code.zip`, criado a partir dos
  arquivos locais necessários, sem imagens, manifestos, ambientes ou pesos.
- `github`: somente após publicar a branch. Preencha seu nome exato em
  `GITHUB_BRANCH`. Este notebook não publica a branch.
- `existing`: apontar para uma cópia já extraída em `EXISTING_ROOT`.

O ZIP deve ter caminhos `src/...`, `tests/...` na sua raiz. A extração usa uma
lista permitida de arquivos e um diretório temporário novo. GitHub é um download
de código autorizado pela escolha dessa opção; não baixa o dataset.

In [ ]:
import hashlib
import io
import os
import stat
import tempfile
import zipfile

CODE_SOURCE = "upload_zip"  # upload_zip, github ou existing
GITHUB_URL = "https://github.com/Caio-crl/breast-cancer-ml-classification.git"
GITHUB_BRANCH = ""  # preencher somente depois de publicar a branch de Computer Vision
EXISTING_ROOT = Path("/content/tech-challenge-cv")
CODE_FILES = {
    "src/__init__.py", "src/vision/__init__.py", "src/vision/inspect_dataset.py",
    "src/vision/prepare_manifest.py", "src/vision/dataset.py", "src/vision/model.py",
    "src/vision/smoke_test.py", "tests/vision/test_inspect_dataset.py",
    "tests/vision/test_prepare_manifest.py", "tests/vision/test_dataset.py",
    "tests/vision/test_model.py", "requirements-vision.txt", "docs/vision_infrastructure.md",
}

if CODE_SOURCE == "upload_zip":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("Envie somente o ZIP de código.")
    payload = next(iter(uploaded.values()))
    print("SHA-256 do ZIP:", hashlib.sha256(payload).hexdigest())
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="tech-challenge-cv6-code-"))
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = [entry for entry in archive.infolist() if not entry.is_dir()]
        names = [entry.filename for entry in entries]
        if len(names) != len(set(names)) or set(names) != CODE_FILES:
            raise ValueError("ZIP diferente da lista de código esperada; não inclua dados ou ambientes.")
        if sum(entry.file_size for entry in entries) > 5_000_000:
            raise ValueError("ZIP de código maior que o limite esperado.")
        for entry in entries:
            if stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Links simbólicos não são permitidos no ZIP.")
            destination = PROJECT_ROOT / entry.filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(entry))
    del uploaded, payload
elif CODE_SOURCE == "github":
    if not GITHUB_BRANCH.strip() or GITHUB_BRANCH.startswith("-"):
        raise ValueError("Informe a branch publicada; nenhum nome foi presumido.")
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="tech-challenge-cv6-git-"))
    subprocess.run(["git", "clone", "--single-branch", "--branch", GITHUB_BRANCH,
                    GITHUB_URL, str(PROJECT_ROOT)], check=True)
    print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"],
                                             cwd=PROJECT_ROOT, text=True).strip())
elif CODE_SOURCE == "existing":
    PROJECT_ROOT = EXISTING_ROOT.expanduser().resolve()
else:
    raise ValueError("CODE_SOURCE inválido.")

missing = sorted(name for name in CODE_FILES if not (PROJECT_ROOT / name).is_file())
if missing:
    raise FileNotFoundError("Código CV-5 incompleto: " + repr(missing))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Raiz do código:", PROJECT_ROOT)
source_hashes = {name: hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest()
                 for name in sorted(CODE_FILES)}
print("Identificação dos fontes:", json.dumps(source_hashes, indent=2))

## 3. Dependências — manter o runtime existente se estiver funcional

Se a inspeção passou, esta célula não executa pip. Caso falte uma biblioteca,
leia os erros antes de habilitar `INSTALL_IF_NEEDED`. Instalação é opcional e
restrita ao runtime Colab. Não execute esta célula para instalar no Windows.

O par de referência PyTorch 2.7.1 / torchvision 0.22.1 consta na
[matriz oficial](https://pytorch.org/get-started/previous-versions/#v2-7-1).
Outros pares já funcionais podem ser mantidos. NumPy, pandas, Pillow e scikit-learn
são necessários para executar **toda** a suíte CV-2 a CV-5. Não é necessário
instalar JupyterLab, matplotlib ou torchaudio para este notebook.

O fallback padrão usa wheels CPU: não depende de GPU ou de versão CUDA presumida.
Se houver instalação, reinicie a sessão e execute novamente desde o início.

In [ ]:
INSTALL_IF_NEEDED = False
USE_CPU_WHEELS = True  # suficiente para testes e forward; GPU pode usar a pilha funcional já fornecida
environment = probe_environment()
if not environment["ok"]:
    if not INSTALL_IF_NEEDED:
        raise RuntimeError("Dependências incompatíveis/ausentes. Revise os erros acima; "
                           "habilite INSTALL_IF_NEEDED apenas se necessário no Colab.")
    if "google.colab" not in sys.modules:
        import importlib.util
        if importlib.util.find_spec("google.colab") is None:
            raise RuntimeError("Instalação opcional restrita ao Google Colab.")
    auxiliary = {"numpy": "numpy==1.26.4", "pandas": "pandas==2.2.3",
                 "PIL": "Pillow>=11,<14", "sklearn": "scikit-learn==1.6.1"}
    needed = [requirement for module, requirement in auxiliary.items()
              if module not in environment["dependencies"]]
    if needed:
        subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", *needed], check=True)
    if not environment["torch_compatible"]:
        command = [sys.executable, "-m", "pip", "install", "--force-reinstall", "torch==2.7.1", "torchvision==0.22.1"]
        if USE_CPU_WHEELS:
            command += ["--index-url", "https://download.pytorch.org/whl/cpu"]
        subprocess.run(command, check=True)
    raise RuntimeError("Instalação solicitada concluída. Reinicie a sessão antes de continuar.")
print("Ambiente funcional; nenhuma reinstalação necessária.")

## 4. Executar a suíte de visão e identificar os 22 testes da CV-5

Não há download de pesos. O teste de carregamento local usa um state_dict
aleatório temporário, não um checkpoint treinado. A tabela mostra o resultado
individual dos 15 testes de Dataset e 7 de modelo antes ignorados. Qualquer skip,
erro ou falha impede prosseguir como validação aprovada.

In [ ]:
import unittest
import pandas as pd

class RecordingResult(unittest.TextTestResult):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statuses = {}
    def addSuccess(self, test):
        super().addSuccess(test)
        self.statuses[test.id()] = "aprovado"
    def addSkip(self, test, reason):
        super().addSkip(test, reason)
        self.statuses[test.id()] = "ignorado: " + reason
    def addFailure(self, test, err):
        super().addFailure(test, err)
        self.statuses[test.id()] = "falha"
    def addError(self, test, err):
        super().addError(test, err)
        self.statuses[test.id()] = "erro"

suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / "tests/vision"), pattern="test_*.py")
test_result = unittest.TextTestRunner(verbosity=2, resultclass=RecordingResult).run(suite)
previously_skipped = pd.DataFrame([
    {"teste": name, "resultado": status}
    for name, status in sorted(test_result.statuses.items())
    if name.startswith(("test_dataset.DatasetTests.", "test_model.ModelTests."))
])
display(previously_skipped)
test_summary = {"discovered": test_result.testsRun,
                "approved": sum(status == "aprovado" for status in test_result.statuses.values()),
                "skipped": len(test_result.skipped), "failures": len(test_result.failures),
                "errors": len(test_result.errors)}
print(test_summary)
assert test_result.wasSuccessful() and not test_result.skipped, "Há falhas ou skips pendentes."
assert len(previously_skipped) == 22, "Os 22 testes esperados não foram identificados; confira a versão do código."
assert previously_skipped.resultado.eq("aprovado").all(), "Nem todos os 22 testes foram aprovados."

## 5. Integração sintética temporária

Cria nove JPGs em escala de cinza, três pacientes distintos por partição e os
manifestos necessários ao Dataset. Todos os rótulos e arquivos são artificiais.
Nada é gravado no dataset real, no Drive ou nos manifestos CV-4.

In [ ]:
import numpy as np
from PIL import Image
from src.vision.prepare_manifest import COLUMNS

synthetic_temp = tempfile.TemporaryDirectory(prefix="cv6-synthetic-")
SYNTHETIC_ROOT = Path(synthetic_temp.name) / "cbis_ddsm"
SYNTHETIC_MANIFESTS = Path(synthetic_temp.name) / "manifests"
SYNTHETIC_MANIFESTS.mkdir(parents=True)
synthetic_original_hashes = {}
for split_index, split in enumerate(("train", "val", "test")):
    rows = []
    for index, pathology in enumerate(("BENIGN", "MALIGNANT", "BENIGN_WITHOUT_CALLBACK")):
        number = split_index * 3 + index + 1
        patient = f"P_{number:05d}"
        prefix = "Test" if split == "test" else "Training"
        relative = f"jpeg/1.2.{number}/crop.jpg"
        path = SYNTHETIC_ROOT / relative
        path.parent.mkdir(parents=True)
        pixels = np.random.default_rng(number).integers(0, 256, (40 + number, 80 + number), dtype=np.uint8)
        with Image.fromarray(pixels) as image:
            width, height = image.size
            image.save(path, format="JPEG")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        synthetic_original_hashes[relative] = digest
        rows.append({"patient_id": patient, "case_id": f"Mass-{prefix}_{patient}_LEFT_CC_1",
                     "image_path": relative, "pathology": pathology,
                     "label": int(pathology == "MALIGNANT"), "split": split,
                     "official_split": "test" if split == "test" else "train",
                     "width": width, "height": height, "mode": "L", "sha256": digest})
    pd.DataFrame(rows, columns=COLUMNS).to_csv(SYNTHETIC_MANIFESTS / f"manifest_{split}.csv", index=False)
print("Fixtures sintéticas:", SYNTHETIC_ROOT, SYNTHETIC_MANIFESTS)

## 6. Dataset, transformações e DataLoaders

Usa o código do projeto. Confere RGB, proporção/padding, normalização e avaliação
determinística. O treino mantém augmentation; validação e teste não usam rotação.

In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import RandomSampler, SequentialSampler
from src.vision.dataset import build_dataloaders, build_transforms, Letterbox
from src.vision.model import build_model, parameter_counts

torch.manual_seed(42)
np.random.seed(42)
synthetic_loaders = build_dataloaders(SYNTHETIC_MANIFESTS, SYNTHETIC_ROOT, batch_size=2, num_workers=0, seed=42)
assert isinstance(synthetic_loaders["train"].sampler, RandomSampler)
assert all(isinstance(synthetic_loaders[s].sampler, SequentialSampler) for s in ("val", "test"))
assert any(isinstance(t, transforms.RandomRotation) for t in build_transforms(True).transforms)
assert not any(isinstance(t, transforms.RandomRotation) for t in build_transforms(False).transforms)
validation_dataset = synthetic_loaders["val"].dataset
assert torch.equal(validation_dataset[0][0], validation_dataset[0][0])
for split, loader in synthetic_loaders.items():
    images, labels = next(iter(loader))
    assert tuple(images.shape) == (2, 3, 224, 224)
    assert images.dtype == torch.float32 and labels.dtype == torch.int64
    assert torch.isfinite(images).all() and set(labels.tolist()).issubset({0, 1})
    print(split, "imagens:", tuple(images.shape), "rótulos:", labels.tolist())

with Image.new("RGB", (80, 40), (255, 0, 0)) as rectangle:
    with Letterbox()(rectangle) as padded:
        assert padded.getchannel("R").getbbox() == (0, 56, 224, 168)
with Image.new("RGB", (80, 40), (128, 128, 128)) as constant:
    normalized = build_transforms(False)(constant)
expected = torch.tensor([(128 / 255 - mean) / std for mean, std in zip(
    (0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
assert torch.allclose(normalized[:, 112, 112], expected, atol=1e-5)
print("Proporção, padding, normalização e separação das transformações: conferidos.")

## 7. Smoke test sintético: um forward, sem gradientes

CPU por padrão. Para GPU, escolha `auto` ou `cuda`; CUDA explicitamente solicitada
e indisponível gera erro. A rede usa **weights=None** (`pretrained=False`).
Congelamos as features apenas para verificar o comportamento técnico; isso não é
uma recomendação de treinar features aleatórias congeladas. Os testes também
conferem o modelo aleatório totalmente treinável e o carregamento local de pesos.

In [ ]:
DEVICE_REQUEST = "cpu"  # cpu, auto ou cuda
if DEVICE_REQUEST not in {"cpu", "auto", "cuda"}:
    raise ValueError("Dispositivo inválido.")
if DEVICE_REQUEST == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA solicitada, mas indisponível.")
DEVICE = torch.device("cuda" if DEVICE_REQUEST != "cpu" and torch.cuda.is_available() else "cpu")
smoke_results = []

def run_one_batch(loaders, source):
    torch.manual_seed(42)
    images, labels = next(iter(loaders["train"]))
    assert tuple(images.shape[1:]) == (3, 224, 224)
    assert torch.isfinite(images).all()
    assert labels.dtype == torch.int64 and set(labels.tolist()).issubset({0, 1})
    model = build_model(pretrained=False, freeze_features=True).to(DEVICE).eval()
    counts = parameter_counts(model)
    assert not any(p.requires_grad for p in model.features.parameters())
    assert all(p.requires_grad for p in model.classifier.parameters())
    assert all(not block.training for block in model.features)
    with torch.inference_mode():
        logits = model(images.to(DEVICE))
    assert tuple(logits.shape) == (len(labels), 2)
    assert torch.isfinite(logits).all() and not logits.requires_grad
    result = {"source": source, "device": str(DEVICE), "input_shape": list(images.shape),
              "logits_shape": list(logits.shape), "finite": True, "weights": None,
              "total_parameters": counts["total"], "trainable_parameters": counts["trainable"],
              "features_frozen": True, "executed": True}
    print(json.dumps(result, indent=2))
    smoke_results.append(result)
    return result

synthetic_result = run_one_batch(synthetic_loaders, "synthetic")
assert all(hashlib.sha256((SYNTHETIC_ROOT / relative).read_bytes()).hexdigest() == digest
           for relative, digest in synthetic_original_hashes.items())

## 8. Opcional: repetir com dados reais no Google Drive

O Git não fornece dataset nem manifestos. **Não se presume que já estejam no
Drive.** Depois de transferi-los manualmente, ajuste os dois caminhos abaixo.
Preserve `jpeg/<UID>/<arquivo>.jpg` sob a raiz do CBIS-DDSM e os três manifestos
CV-4, sem recriar a separação ou alterar os CSVs. Basta transferir os recortes
referenciados para integração, desde que todos os caminhos dos manifestos existam.
Com os manifestos completos isso significa disponibilizar os 1.696 recortes.

O lote real só será carregado se `USE_REAL_DATA=True` e após o smoke sintético.
Não será feita nova auditoria completa de pixels/hashes nem avaliação preditiva.

In [ ]:
USE_REAL_DATA = False
REAL_DATA_ROOT = Path("/content/drive/MyDrive/tech-challenge/cbis_ddsm")
REAL_MANIFEST_DIR = Path("/content/drive/MyDrive/tech-challenge/processed")

if USE_REAL_DATA:
    assert synthetic_result["executed"], "Execute primeiro o smoke sintético."
    from google.colab import drive
    drive.mount("/content/drive")
    if not REAL_DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Dataset ainda não acessível em {REAL_DATA_ROOT}")
    for split in ("train", "val", "test"):
        if not (REAL_MANIFEST_DIR / f"manifest_{split}.csv").is_file():
            raise FileNotFoundError(f"Manifesto {split} ainda não acessível em {REAL_MANIFEST_DIR}")
    real_loaders = build_dataloaders(REAL_MANIFEST_DIR, REAL_DATA_ROOT, batch_size=2, num_workers=0, seed=42)
    real_result = run_one_batch(real_loaders, "real")
else:
    print("Integração real não executada: USE_REAL_DATA=False. Nenhum acesso ao Drive solicitado.")

## 9. Resultado desta sessão

Somente resultados produzidos pelas células anteriores são apresentados. Salve
uma cópia do notebook com as saídas para documentar versões, fontes, os 22 testes
antes ignorados, dispositivo, formatos, finitude e parâmetros. Não apresente
aprovação de integração real quando apenas o cenário sintético foi executado.

Antes do treinamento: resolver qualquer falha, executar o lote real quando
acessível e decidir explicitamente como disponibilizar pesos ImageNet locais.
Não há épocas, otimizador ou checkpoints treinados neste notebook.

In [ ]:
assert test_summary["skipped"] == 0 and test_result.wasSuccessful()
assert synthetic_result["finite"] and synthetic_result["executed"]
print("Testes nesta sessão:", test_summary)
print("Dos 22 testes antes ignorados:", int(previously_skipped.resultado.eq("aprovado").sum()), "aprovados")
display(pd.DataFrame(smoke_results))
print("Integração real:", "executada" if any(r["source"] == "real" for r in smoke_results) else "pendente")
print("Sem treinamento; logits aleatórios não representam desempenho ou diagnóstico clínico.")